# OpenJev dataset exploration
Explore the OpenJev decision datasets before training a Gemma 4 E2B decision model.
Files are JSONL named `<source>.<split>.jsonl`. Each record has `id, source, domain, state, questions, answers, weight`.

In [ ]:
import glob, os, json, hashlib, random
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

MAX_ROWS_PER_FILE = 50_000   # cap per file for speed
MAX_STATE_TOKENS = 1024      # our training default
HARD_TOKENS = 4096

files = sorted(glob.glob("/kaggle/input/**/*.jsonl", recursive=True))
if not files:
    files = sorted(glob.glob("../data/*.jsonl"))
print(len(files), "files; first:", files[:1])
assert files, "no JSONL files found"

def iter_jsonl(path, limit):
    """Stream a JSONL file line by line, stopping after `limit` rows."""
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= limit:
                break
            line = line.strip()
            if line:
                yield json.loads(line)

## 1. Load + rows per source/split

In [ ]:
data = {}  # (source, split) -> list[dict]
for p in files:
    parts = os.path.basename(p).split(".")
    if len(parts) != 3 or parts[1] not in ("train", "val", "test"):
        continue
    data[(parts[0], parts[1])] = list(iter_jsonl(p, MAX_ROWS_PER_FILE))

SOURCES = sorted({s for s, _ in data})
SPLITS = ["train", "val", "test"]
rows = pd.DataFrame(
    [[len(data.get((s, sp), [])) for sp in SPLITS] for s in SOURCES],
    index=SOURCES, columns=SPLITS)
rows["total"] = rows.sum(axis=1)
print(f"(capped at {MAX_ROWS_PER_FILE:,} rows per file)")
rows

## 2. Question types per source; options per question

In [ ]:
qrecs = []  # one row per (record, question)
for (s, sp), recs in data.items():
    for r in recs:
        for qn, q in r["questions"].items():
            crit = q.get("criteria")
            n_opts = len(crit) if crit is not None else 0
            qrecs.append((s, sp, qn, q["type"], n_opts))
qdf = pd.DataFrame(qrecs, columns=["source", "split", "qname", "qtype", "n_opts"])

per_src = qdf.groupby(["source", "qname", "qtype"]).agg(
    n=("n_opts", "size"), min_opts=("n_opts", "min"),
    median_opts=("n_opts", "median"), max_opts=("n_opts", "max"))
display(per_src)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, t in zip(axes, ["noul", "choice", "score"]):
    v = qdf.loc[qdf.qtype == t, "n_opts"]
    if len(v):
        ax.hist(v, bins=min(50, max(2, int(v.max() - v.min()) + 1)))
    ax.set_title(f"options per question: {t}")
    ax.set_xlabel("# options"); ax.set_ylabel("questions")
plt.tight_layout(); plt.show()

## 3. State length (chars and approx tokens = chars/4)
Gemma 4 sliding window is 512 tokens; our max-state-tokens default is 1024, so anything longer is truncated at training time.

In [ ]:
lens = pd.DataFrame(
    [(s, sp, len(r["state"])) for (s, sp), recs in data.items() for r in recs],
    columns=["source", "split", "chars"])
lens["tokens"] = lens["chars"] / 4

def p95(x): return np.percentile(x, 95)
stat = lens.groupby("source").agg(
    rows=("chars", "size"),
    chars_median=("chars", "median"), chars_p95=("chars", p95), chars_max=("chars", "max"),
    tok_median=("tokens", "median"), tok_p95=("tokens", p95), tok_max=("tokens", "max"))
stat["over_1024_tok"] = lens.groupby("source")["tokens"].apply(lambda x: float((x > MAX_STATE_TOKENS).sum()))
stat["pct_truncated_1024"] = (100 * stat["over_1024_tok"] / stat["rows"]).astype(float).round(2)
stat["over_4096_tok"] = lens.groupby("source")["tokens"].apply(lambda x: float((x > HARD_TOKENS).sum()))
stat["pct_over_4096"] = (100 * stat["over_4096_tok"] / stat["rows"]).astype(float).round(2)
display(stat.round(1))
print(f"Total rows truncated at {MAX_STATE_TOKENS} approx tokens: {int((lens.tokens > MAX_STATE_TOKENS).sum()):,} of {len(lens):,}")
print(f"Total rows over {HARD_TOKENS} approx tokens (flag): {int((lens.tokens > HARD_TOKENS).sum()):,}")

fig, ax = plt.subplots(figsize=(10, 4))
groups = [lens.loc[lens.source == s, "tokens"].values for s in SOURCES]
ax.boxplot(groups, showfliers=False)
ax.set_xticks(range(1, len(SOURCES) + 1)); ax.set_xticklabels(SOURCES)
ax.axhline(MAX_STATE_TOKENS, color="r", ls="--", label="max-state-tokens 1024")
ax.axhline(512, color="orange", ls=":", label="sliding window 512")
ax.set_ylabel("approx tokens (chars/4)"); ax.set_yscale("log")
ax.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

## 4. Label balance and majority-class baseline

In [ ]:
def norm_label(q, a):
    """Normalise an answer to a string label (score index -> level name if possible)."""
    if a is None:
        return None
    if q["type"] == "score":
        crit = q.get("criteria") or []
        if isinstance(a, int) and not isinstance(a, bool) and 0 <= a < len(crit):
            return f"{a}:{crit[a]}" if isinstance(crit[a], str) and len(crit[a]) < 30 else str(a)
        if isinstance(a, str) and a in crit:
            i = crit.index(a)
            return f"{i}:{a}" if len(a) < 30 else str(i)
    return str(a)

labels = defaultdict(lambda: defaultdict(Counter))  # (source,qname) -> split -> Counter
nulls = Counter()
for (s, sp), recs in data.items():
    for r in recs:
        for qn, q in r["questions"].items():
            a = r["answers"].get(qn)
            if a is None:
                nulls[(s, qn, sp)] += 1
            else:
                labels[(s, qn)][sp][norm_label(q, a)] += 1

out = []
for (s, qn), by_sp in sorted(labels.items()):
    tr, va = by_sp["train"], by_sp["val"]
    ntr = sum(tr.values()); nva = sum(va.values())
    top_lab, top_n = (tr.most_common(1)[0] if tr else (None, 0))
    base = (va.get(top_lab, 0) / nva) if nva else np.nan
    out.append(dict(source=s, question=qn, n_labels=len(tr), train_n=ntr, top_label=str(top_lab)[:30],
                    top_share_train=round(top_n / ntr, 3) if ntr else np.nan,
                    majority_acc_val=round(base, 3),
                    null_train=nulls[(s, qn, "train")], null_val=nulls[(s, qn, "val")],
                    null_test=nulls[(s, qn, "test")]))
bal = pd.DataFrame(out)
bal

## 5. Leak check
For choice questions: how often does the correct option text appear verbatim in `state`, versus a random wrong option? A large gap means the model can shortcut by string matching (STATS.md reports pokerbench at +0.38).

In [ ]:
rng = random.Random(0)
SAMPLE = 5000
leak = []
for s in SOURCES:
    recs = [r for sp in SPLITS for r in data.get((s, sp), [])]
    rng.shuffle(recs)
    qnames = sorted({qn for r in recs[:SAMPLE] for qn, q in r["questions"].items() if q["type"] == "choice"})
    for qn in qnames:
        hit_c = hit_w = n = 0
        for r in recs[:SAMPLE]:
            q = r["questions"].get(qn)
            a = r["answers"].get(qn)
            if not q or q["type"] != "choice" or a is None or a not in q["criteria"]:
                continue
            wrong = [o for o in q["criteria"] if o != a]
            if not wrong:
                continue
            n += 1
            hit_c += a in r["state"]
            hit_w += rng.choice(wrong) in r["state"]
        if n:
            leak.append(dict(source=s, question=qn, n=n, correct_in_state=round(hit_c / n, 3),
                             wrong_in_state=round(hit_w / n, 3), gap=round((hit_c - hit_w) / n, 3)))
leak_df = pd.DataFrame(leak)
display(leak_df.sort_values("gap", ascending=False) if len(leak_df) else leak_df)

## 6. Score questions: level distribution

In [ ]:
sc = []
for (s, sp), recs in data.items():
    for r in recs:
        for qn, q in r["questions"].items():
            if q["type"] != "score":
                continue
            a = r["answers"].get(qn)
            if a is None:
                continue
            crit = q["criteria"]
            if isinstance(a, str) and a in crit:
                a = crit.index(a)
            sc.append((s, qn, int(a)))
scdf = pd.DataFrame(sc, columns=["source", "question", "level"])
if len(scdf):
    dist = scdf.groupby(["source", "question"])["level"].value_counts(normalize=True).unstack(fill_value=0).round(3)
    display(dist)
    keys = list(dist.index)
    fig, axes = plt.subplots(1, len(keys), figsize=(4 * len(keys), 3), squeeze=False)
    for ax, k in zip(axes[0], keys):
        ax.bar([str(c) for c in dist.columns], dist.loc[k].values); ax.set_title(f"{k[0]}/{k[1]}"); ax.set_xlabel("level"); ax.set_ylabel("share")
    plt.tight_layout(); plt.show()
else:
    print("no score questions found")

## 7. Duplicate `state` across train/val/test

In [ ]:
def h(s): return hashlib.md5(s.encode("utf-8")).digest()
dup = []
for s in SOURCES:
    hs = {sp: [h(r["state"]) for r in data.get((s, sp), [])] for sp in SPLITS}
    sets = {sp: set(v) for sp, v in hs.items()}
    dup.append(dict(
        source=s,
        dup_within_train=len(hs["train"]) - len(sets["train"]),
        train_val=len(sets["train"] & sets["val"]),
        train_test=len(sets["train"] & sets["test"]),
        val_test=len(sets["val"] & sets["test"]),
        val_in_train_pct=round(100 * len(sets["val"] & sets["train"]) / max(1, len(sets["val"])), 2),
        test_in_train_pct=round(100 * len(sets["test"] & sets["train"]) / max(1, len(sets["test"])), 2)))
pd.DataFrame(dup).set_index("source")

## 8. Random example records (2 per source, state truncated to 400 chars)

In [ ]:
rng = random.Random(42)
for s in SOURCES:
    pool = [r for sp in SPLITS for r in data.get((s, sp), [])]
    for r in rng.sample(pool, min(2, len(pool))):
        r = dict(r)
        r["state"] = r["state"][:400] + ("..." if len(r["state"]) > 400 else "")
        qs = {}
        for qn, q in r["questions"].items():
            q = dict(q)
            if isinstance(q.get("criteria"), dict) and len(q["criteria"]) > 8:
                keys = list(q["criteria"])
                q["criteria"] = {k: q["criteria"][k] for k in keys[:8]}
                q["criteria"]["..."] = f"({len(keys) - 8} more options)"
            qs[qn] = q
        r["questions"] = qs
        print("=" * 100); print(s)
        print(json.dumps(r, indent=2, ensure_ascii=False))

## 9. Summary: what to watch for when training
- **Truncation:** states longer than the max-state-tokens setting are cut; check section 3 for sources where a notable share is truncated, and consider truncating from the left/middle so the decision-relevant tail (question, latest move) survives.
- **Sliding window:** Gemma 4 uses a short local window, so long states depend on global layers; sources with long states may train and generalise differently.
- **Class imbalance:** compare accuracy to the majority-class baseline per question (section 4); report macro-F1 as well as accuracy, and consider class weighting or resampling for skewed labels.
- **Unlabeled answers:** null answers must be masked out of the loss and out of metrics.
- **Leakage:** a positive gap in section 5 means the model can score by string matching option text in the state instead of reasoning; evaluate with the leaky option text masked or shuffled to see how much it matters.
- **Score questions:** skewed level distributions (section 6) make a constant prediction look good; check per-level recall and use ordinal-aware metrics where sensible.
- **Duplicates:** shared `state` strings between train and val/test (section 7) inflate eval scores; deduplicate or report scores with the overlap removed.
- **Mixed sources:** very different sizes means large sources dominate batches; use per-source sampling weights or temperature sampling and track per-source metrics, not just the aggregate.
- **Caps:** this notebook caps each file at `MAX_ROWS_PER_FILE`; statistics for larger files are computed on the head of the file, which may not be representative if files are ordered.